# ⚛️ Kuantum Kapıları — Kapsamlı Rehber

**Hazırlayan:** Dr. Buket Toptaş 
**Bölüm:** Yazılım Mühendisliği 
**Ders:** Kuantum Makine Öğrenmesi (QML)

---

## Bu Not Defterinde Neler Öğreneceksin?

| Bölüm | Konu | Kapı |
|-------|------|------|
| 1 | Kurulum & Temel Kavramlar | — |
| 2 | Birim (Identity) Kapısı | I |
| 3 | Pauli-X (NOT) Kapısı | X |
| 4 | Pauli-Z (Faz) Kapısı | Z |
| 5 | Pauli-Y Kapısı | Y |
| 6 | Hadamard Kapısı | H |
| 7 | S ve T Faz Kapıları | S, T |
| 8 | Döndürme Kapıları | Rx, Ry, Rz |
| 9 | CNOT (Kontrollü-NOT) | CX |
| 10 | SWAP Kapısı | SWAP |
| 11 | Toffoli (CCX) Kapısı | CCX |
| 12 | Bell Durumu & Dolanıklık | H + CNOT |
| 13 | Kapı Özet Tablosu | Hepsi |

Her kapı için:
- 📐 Matris gösterimi
- 🔵 Bloch küresi görselleştirmesi
- 🔌 Devre şeması
- 📊 Ölçüm sonuçları
- 💡 Sade Türkçe açıklama

---
## 1. Kurulum ve Temel Kavramlar

Önce gerekli kütüphaneleri kuralım.

In [ ]:
# Kütüphaneleri kur
!pip install qiskit qiskit-aer pylatexenc -q

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# Qiskit modülleri
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_bloch_multivector, plot_histogram

print('✅ Kurulum tamamlandı!')
print(f'📦 Qiskit hazır')

### Qubit Nedir? (30 saniyede)

Klasik bilgisayarda **bit** = 0 veya 1. Kuantum bilgisayarda **qubit** = 0, 1 veya **ikisinin karışımı** (süperpozisyon).

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

- $\alpha$ ve $\beta$ karmaşık sayılar
- $|\alpha|^2$ = 0 ölçme olasılığı
- $|\beta|^2$ = 1 ölçme olasılığı
- $|\alpha|^2 + |\beta|^2 = 1$ (toplam olasılık = %100)

### Kuantum Kapısı Nedir?

Qubit'in durumunu **değiştiren işlem**. Matematikte bir **matris**, devrede bir **kutu**.

In [ ]:
# Yardımcı fonksiyonlar — tüm not defteri boyunca kullanacağız

def goster_kapi(kapi_adi, aciklama, matris_str, qc, giris='|0⟩', beklenen=''):
    """Bir kuantum kapısını görselleştir: devre + Bloch küresi + ölçüm."""
    print('=' * 60)
    print(f'🔷 {kapi_adi}')
    print(f'   {aciklama}')
    print(f'   Matris: {matris_str}')
    print(f'   Giriş: {giris} → Beklenen çıkış: {beklenen}')
    print('=' * 60)

    # Durum vektörünü hesapla
    sv = Statevector.from_instruction(qc)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # 1) Devre şeması
    axes[0].set_title('Devre Şeması', fontsize=13, fontweight='bold', color='#1E293B')
    qc.draw('mpl', ax=axes[0])

    # 2) Bloch küresi
    axes[1].set_title('Bloch Küresi', fontsize=13, fontweight='bold', color='#1E293B')
    # Bloch küresi ayrı figürde çizilir, onu embed edeceğiz
    axes[1].axis('off')
    axes[1].text(0.5, 0.5, '(Aşağıda)', ha='center', va='center', fontsize=12, color='gray',
                transform=axes[1].transAxes)

    # 3) Olasılık çubukları
    probs = sv.probabilities_dict()
    states = sorted(probs.keys())
    values = [probs[s] for s in states]
    colors = ['#0891B2' if v < 0.5 else '#10B981' for v in values]
    bars = axes[2].bar(states, values, color=colors, edgecolor='#1E293B', linewidth=1.2)
    axes[2].set_ylim(0, 1.15)
    axes[2].set_title('Ölçüm Olasılıkları', fontsize=13, fontweight='bold', color='#1E293B')
    axes[2].set_ylabel('Olasılık')
    for bar, val in zip(bars, values):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                    f'%{val*100:.0f}', ha='center', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Bloch küresi (ayrı figür)
    display(plot_bloch_multivector(sv))

    # Durum vektörü
    print(f'\n📊 Durum vektörü: {sv}')
    print()


def matris_goster(kapi_adi, matris):
    """Matris güzel şekilde göster."""
    print(f'\n📐 {kapi_adi} Matrisi:')
    print(np.round(matris, 3))
    print()

print('✅ Yardımcı fonksiyonlar hazır!')

### Başlangıç: |0⟩ ve |1⟩ durumları

Kapıları görmeden önce, qubit'in iki temel durumunu tanıyalım.

In [ ]:
# |0⟩ durumu — Bloch küresinin Kuzey Kutbu
sv_0 = Statevector.from_label('0')
print('|0⟩ durumu:', sv_0)
print('Vektör: [1, 0] → "kesinlikle 0"')
print()

# |1⟩ durumu — Bloch küresinin Güney Kutbu
sv_1 = Statevector.from_label('1')
print('|1⟩ durumu:', sv_1)
print('Vektör: [0, 1] → "kesinlikle 1"')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax in axes:
    ax.axis('off')
axes[0].set_title('|0⟩ = Kuzey Kutbu', fontsize=14, fontweight='bold', color='#0891B2')
axes[1].set_title('|1⟩ = Güney Kutbu', fontsize=14, fontweight='bold', color='#EF4444')
plt.tight_layout()
plt.show()

display(plot_bloch_multivector(sv_0))
display(plot_bloch_multivector(sv_1))

---
## 2. Birim (Identity) Kapısı — I

**Ne yapar?** Hiçbir şey! Qubit'i olduğu gibi bırakır.

**Neden var?** Matematiksel tamlık ve devre zamanlama için gerekli.

$$I = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}$$

**Analoji:** Bir kağıdı fotokopi makinesine koyup aynısını almak.

In [ ]:
# Identity kapısı
qc = QuantumCircuit(1)  # 1 qubitlik devre
qc.id(0)               # Identity uygula

goster_kapi(
    'Identity (I) Kapısı',
    'Hiçbir şey yapmaz — qubit olduğu gibi kalır.',
    '[[1,0],[0,1]]',
    qc,
    giris='|0⟩',
    beklenen='|0⟩ (değişmedi)'
)

matris_goster('Identity (I)', Operator(qc).data)

---
## 3. Pauli-X Kapısı (NOT / Bit-Flip)

**Ne yapar?** 0'ı 1'e, 1'i 0'a çevirir. Klasik NOT kapısı gibi.

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$$

$$X|0\rangle = |1\rangle, \quad X|1\rangle = |0\rangle$$

**Analoji:** Işık anahtarı — açıksa kapat, kapalıysa aç.

**Bloch küresinde:** X ekseni etrafında 180° döndürme (Kuzey ↔ Güney).

**İsim kökeni:** Wolfgang Pauli (1900-1958), Nobel ödüllü Avusturyalı fizikçi.

In [ ]:
# Pauli-X: |0⟩ → |1⟩
qc = QuantumCircuit(1)
qc.x(0)  # X kapısı uygula

goster_kapi(
    'Pauli-X (NOT) Kapısı',
    '|0⟩ → |1⟩ çevirir. Klasik NOT gibi.',
    '[[0,1],[1,0]]',
    qc,
    giris='|0⟩',
    beklenen='|1⟩'
)

matris_goster('Pauli-X', Operator(qc).data)

In [ ]:
# X kapısını iki kez uygula: |0⟩ → |1⟩ → |0⟩ (geri döner!)
qc_xx = QuantumCircuit(1)
qc_xx.x(0)
qc_xx.x(0)

sv_xx = Statevector.from_instruction(qc_xx)
print('X kapısı 2 kez uygulanırsa:')
print(f'|0⟩ → X → |1⟩ → X → |0⟩')
print(f'Sonuç: {sv_xx}')
print(f'Başa döndük! X² = I (iki kez çevir = hiçbir şey yapma)')
display(qc_xx.draw('mpl'))

---
## 4. Pauli-Z Kapısı (Faz Çevirme)

**Ne yapar?** |0⟩'ı olduğu gibi bırakır, |1⟩'in **işaretini** (fazını) değiştirir.

$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

$$Z|0\rangle = |0\rangle, \quad Z|1\rangle = -|1\rangle$$

**Analoji:** Dalganın tepesini çukura, çukurunu tepeye çevirir (180° faz kaydırma).

**Bloch küresinde:** Z ekseni etrafında 180° döndürme.

In [ ]:
# Z kapısı |0⟩'a uygulanırsa — bir şey değişmez!
qc_z0 = QuantumCircuit(1)
qc_z0.z(0)

goster_kapi(
    'Pauli-Z: |0⟩ durumuna uygulama',
    'Z|0⟩ = |0⟩ → Hiçbir değişiklik yok!',
    '[[1,0],[0,-1]]',
    qc_z0,
    giris='|0⟩',
    beklenen='|0⟩ (aynı)'
)

In [ ]:
# Z kapısının etkisini görmek için önce süperpozisyona geçelim
# |0⟩ → H → |+⟩ → Z → |−⟩
qc_hz = QuantumCircuit(1)
qc_hz.h(0)  # Hadamard: süperpozisyona al
qc_hz.z(0)  # Z: fazı çevir

goster_kapi(
    'Pauli-Z: |+⟩ durumuna uygulama',
    'H|0⟩ = |+⟩, sonra Z|+⟩ = |−⟩ → Ekvator üzerinde 180° döndü!',
    '[[1,0],[0,-1]]',
    qc_hz,
    giris='|0⟩ → H → |+⟩',
    beklenen='|−⟩'
)

print('💡 Z kapısı |0⟩ ve |1⟩ da görünmez ama süperpozisyonda FAZ değiştirir.')
print('   Bu faz farkı, kuantum girişiminde çok önemli!')

---
## 5. Pauli-Y Kapısı

**Ne yapar?** Hem bit çevirir (X gibi) hem faz çevirir (Z gibi). İkisinin kombinasyonu.

$$Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$$

$$Y|0\rangle = i|1\rangle, \quad Y|1\rangle = -i|0\rangle$$

**Analoji:** Hem ışık anahtarını çevir, hem dalganın fazını değiştir.

**Bloch küresinde:** Y ekseni etrafında 180° döndürme.

In [ ]:
# Pauli-Y
qc_y = QuantumCircuit(1)
qc_y.y(0)

goster_kapi(
    'Pauli-Y Kapısı',
    'Y|0⟩ = i|1⟩ → Hem bit çevirdi hem faz ekledi!',
    '[[0,-i],[i,0]]',
    qc_y,
    giris='|0⟩',
    beklenen='i|1⟩'
)

matris_goster('Pauli-Y', Operator(qc_y).data)
print('💡 Y = iXZ → X ve Z nin birleşimi (sıra önemli!)')

### 3 Pauli Kapısını Karşılaştıralım

In [ ]:
# 3 Pauli kapısını yan yana karşılaştır
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

paulis = [
    ('Pauli-X (NOT)', 'x', '#EF4444', '|0⟩ ↔ |1⟩\nBit çevirme'),
    ('Pauli-Y', 'y', '#F59E0B', 'Bit + Faz\nçevirme'),
    ('Pauli-Z (Faz)', 'z', '#3B82F6', '|1⟩ → −|1⟩\nFaz çevirme'),
]

for ax, (name, gate, color, desc) in zip(axes, paulis):
    qc = QuantumCircuit(1)
    getattr(qc, gate)(0)  # qc.x(0), qc.y(0), qc.z(0)
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities_dict()
    states = sorted(probs.keys())
    values = [probs[s] for s in states]
    ax.bar(states, values, color=color, edgecolor='#1E293B', linewidth=1.2)
    ax.set_ylim(0, 1.2)
    ax.set_title(f'{name}\n{desc}', fontsize=12, fontweight='bold')
    for i, (s, v) in enumerate(zip(states, values)):
        ax.text(i, v + 0.05, f'%{v*100:.0f}', ha='center', fontsize=14, fontweight='bold')

plt.suptitle('Pauli Kapıları Karşılaştırması (giriş: |0⟩)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 X ve Y, |0⟩ ı kesinlikle |1⟩ e çevirir.')
print('   Z ise |0⟩ da hiçbir değişiklik yapmaz (faz sadece |1⟩ bileşenini etkiler).')

---
## 6. Hadamard Kapısı — H ⭐

**Kuantum hesaplamanın en önemli kapısı!**

**Ne yapar?** |0⟩'ı |0⟩ ve |1⟩'in eşit karışımına (süperpozisyona) sokar.

$$H = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = |+\rangle \quad \text{(%50-%50 süperpozisyon)}$$

$$H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |-\rangle$$

**Analoji:** Bir madeni parayı havaya fırlatmak — yazı mı tura mı belli değil.

**Bloch küresinde:** Kuzey kutbunu (|0⟩) ekvator çizgisine (|+⟩) taşır.

**İsim kökeni:** Jacques Hadamard (1865-1963), Fransız matematikçi.

In [ ]:
# Hadamard: |0⟩ → |+⟩
qc_h = QuantumCircuit(1)
qc_h.h(0)

goster_kapi(
    'Hadamard (H) Kapısı ⭐',
    '|0⟩ → |+⟩ = (|0⟩+|1⟩)/√2 → %50-%50 süperpozisyon!',
    '1/√2 × [[1,1],[1,-1]]',
    qc_h,
    giris='|0⟩',
    beklenen='|+⟩ (%50 |0⟩ + %50 |1⟩)'
)

matris_goster('Hadamard', Operator(qc_h).data)

In [ ]:
# Hadamard: |1⟩ → |−⟩
qc_h1 = QuantumCircuit(1)
qc_h1.x(0)  # Önce |1⟩ yap
qc_h1.h(0)  # Sonra H uygula

goster_kapi(
    'Hadamard: |1⟩ → |−⟩',
    '|1⟩ → |−⟩ = (|0⟩−|1⟩)/√2 → Yine %50-%50 ama FAZ FARKI var!',
    '1/√2 × [[1,1],[1,-1]]',
    qc_h1,
    giris='|1⟩',
    beklenen='|−⟩'
)

print('💡 |+⟩ ve |−⟩ ikisi de %50-%50 verir ama aralarındaki faz farkı')
print('   girişimde (interference) çok önemli rol oynar!')

In [ ]:
# H kapısını iki kez uygula: |0⟩ → |+⟩ → |0⟩ (geri döner!)
qc_hh = QuantumCircuit(1)
qc_hh.h(0)
qc_hh.h(0)

sv_hh = Statevector.from_instruction(qc_hh)
print('H kapısı 2 kez uygulanırsa:')
print(f'|0⟩ → H → |+⟩ → H → |0⟩')
print(f'Sonuç: {sv_hh}')
print(f'Başa döndük! H² = I (Hadamard kendi tersini verir)')
display(qc_hh.draw('mpl'))

---
## 7. S ve T Faz Kapıları

### S Kapısı (√Z)

**Ne yapar?** |1⟩'e 90° (π/2) faz ekler.

$$S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}$$

### T Kapısı (π/8 kapısı)

**Ne yapar?** |1⟩'e 45° (π/4) faz ekler.

$$T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix}$$

**Neden önemli?** {H, T, CNOT} birlikte **evrensel kapı seti** oluşturur — yani herhangi bir kuantum işlemini bu üçüyle yapabilirsin!

In [ ]:
# S ve T kapılarını süperpozisyonda görelim
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Z kapısı: 180° faz
qc_z = QuantumCircuit(1)
qc_z.h(0); qc_z.z(0)
sv_z = Statevector.from_instruction(qc_z)

# S kapısı: 90° faz
qc_s = QuantumCircuit(1)
qc_s.h(0); qc_s.s(0)
sv_s = Statevector.from_instruction(qc_s)

# T kapısı: 45° faz
qc_t = QuantumCircuit(1)
qc_t.h(0); qc_t.t(0)
sv_t = Statevector.from_instruction(qc_t)

print('Faz kapıları karşılaştırması (giriş: |+⟩ = H|0⟩):')
print(f'  Z (180°): {sv_z}')
print(f'  S  (90°): {sv_s}')
print(f'  T  (45°): {sv_t}')
print()
print('💡 Hepsi %50-%50 olasılık verir ama FAZ farklı!')
print('   Bloch küresinin ekvatorunda farklı noktalara düşerler.')

# Bloch küresinde göster
print('\n🔵 Z|+⟩ = |−⟩ (180° faz)')
display(plot_bloch_multivector(sv_z))
print('\n🟢 S|+⟩ (90° faz)')
display(plot_bloch_multivector(sv_s))
print('\n🟡 T|+⟩ (45° faz)')
display(plot_bloch_multivector(sv_t))

---
## 8. Döndürme Kapıları — Rx, Ry, Rz

**QML'de en çok kullanılan kapılar!**

Parametrik kapılardır — açı ($\theta$) alarak qubit'i Bloch küresi üzerinde döndürürler.

$$R_x(\theta) = \begin{pmatrix} \cos\frac{\theta}{2} & -i\sin\frac{\theta}{2} \\ -i\sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{pmatrix}$$

$$R_y(\theta) = \begin{pmatrix} \cos\frac{\theta}{2} & -\sin\frac{\theta}{2} \\ \sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{pmatrix}$$

$$R_z(\theta) = \begin{pmatrix} e^{-i\theta/2} & 0 \\ 0 & e^{i\theta/2} \end{pmatrix}$$

**Neden QML için önemli?** Klasik veriyi qubit'e "yazmak" (kodlamak) için döndürme kapıları kullanılır.

In [ ]:
# Ry kapısını farklı açılarla deneyelim
angles = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]
angle_names = ['0°', '45°', '90°', '135°', '180°']

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

for ax, theta, name in zip(axes, angles, angle_names):
    qc_ry = QuantumCircuit(1)
    qc_ry.ry(theta, 0)
    sv_ry = Statevector.from_instruction(qc_ry)
    probs = sv_ry.probabilities_dict()
    states = sorted(probs.keys())
    values = [probs.get(s, 0) for s in ['0', '1']]
    colors = ['#0891B2', '#7C3AED']
    ax.bar(['|0⟩', '|1⟩'], values, color=colors, edgecolor='#1E293B')
    ax.set_ylim(0, 1.2)
    ax.set_title(f'Ry({name})', fontsize=11, fontweight='bold')
    for i, v in enumerate(values):
        ax.text(i, v + 0.05, f'%{v*100:.0f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Ry Kapısı — Açıya Göre Olasılık Değişimi', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Ry(0°)   = |0⟩ (değişmez)')
print('   Ry(90°)  = |+⟩ benzeri (%50-%50 süperpozisyon)')
print('   Ry(180°) = |1⟩ (tamamen çevrildi = X kapısı gibi)')

In [ ]:
# Bloch küresinde Ry döndürmesini adım adım görelim
print('🔵 Ry kapısı Bloch küresinde Y ekseni etrafında döndürür:')
print()

for theta, name in zip([0, np.pi/4, np.pi/2, np.pi], ['0°', '45°', '90°', '180°']):
    qc_ry = QuantumCircuit(1)
    qc_ry.ry(theta, 0)
    sv = Statevector.from_instruction(qc_ry)
    print(f'Ry({name}): {sv}')
    display(plot_bloch_multivector(sv))

In [ ]:
# QML'de veri kodlama örneği
print('='*60)
print('🧠 QML de Veri Kodlama: Klasik Veri → Qubit')
print('='*60)
print()

# Örnek: 4 özellik değerini 4 qubit'e kodla
veri = [0.5, 1.2, 2.1, 0.8]  # Klasik özellikler

qc_kodlama = QuantumCircuit(4)
for i, x in enumerate(veri):
    qc_kodlama.ry(x, i)  # Her özelliği bir qubit'e Ry ile yaz

print(f'Klasik veri: {veri}')
print(f'Her x değeri, Ry(x) ile bir qubit in açısına yazılır.\n')
display(qc_kodlama.draw('mpl'))
print('\n💡 Bu "angle encoding" yöntemidir. QML nin temel kodlama stratejisi.')

---
## 9. CNOT Kapısı (Kontrollü-NOT) ⭐

**İlk 2-qubit kapımız! Dolanıklığın anahtarı.**

**Ne yapar?** Kontrol qubit'i 1 ise, hedef qubit'i çevirir. 0 ise dokunmaz.

$$\text{CNOT} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$

| Giriş | Çıkış | Açıklama |
|-------|-------|----------|
| \|00⟩ | \|00⟩ | Kontrol=0 → hedef değişmez |
| \|01⟩ | \|01⟩ | Kontrol=0 → hedef değişmez |
| \|10⟩ | \|11⟩ | Kontrol=1 → hedef **çevrildi** |
| \|11⟩ | \|10⟩ | Kontrol=1 → hedef **çevrildi** |

**Analoji:** "Eğer anahtar açıksa (kontrol=1), lambayı çevir (hedef'i NOT)."

In [ ]:
# CNOT: |10⟩ → |11⟩
qc_cx = QuantumCircuit(2)
qc_cx.x(0)      # İlk qubit'i |1⟩ yap (kontrol = 1)
qc_cx.cx(0, 1)   # CNOT: kontrol=q0, hedef=q1

sv_cx = Statevector.from_instruction(qc_cx)

print('='*60)
print('🔷 CNOT (CX) Kapısı')
print('   Kontrol qubit = |1⟩ ise hedefi çevir')
print('='*60)
print(f'\nGiriş: |10⟩ (kontrol=1, hedef=0)')
print(f'Çıkış: {sv_cx}')
print(f'Sonuç: |11⟩ (hedef çevrildi!)\n')

display(qc_cx.draw('mpl'))

# Olasılık grafiği
probs = sv_cx.probabilities_dict()
fig, ax = plt.subplots(figsize=(8, 4))
states = sorted(probs.keys())
values = [probs[s] for s in states]
ax.bar(states, values, color='#7C3AED', edgecolor='#1E293B', linewidth=1.2)
ax.set_ylim(0, 1.2)
ax.set_title('CNOT Çıkış Olasılıkları', fontsize=14, fontweight='bold')
ax.set_ylabel('Olasılık')
for i, (s, v) in enumerate(zip(states, values)):
    ax.text(i, v + 0.05, f'%{v*100:.0f}', ha='center', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CNOT: 4 durumu test et
print('CNOT Doğruluk Tablosu Testi:')
print('-' * 40)

for q0, q1 in [(0,0), (0,1), (1,0), (1,1)]:
    qc = QuantumCircuit(2)
    if q0: qc.x(0)  # Kontrol qubit'i ayarla
    if q1: qc.x(1)  # Hedef qubit'i ayarla
    qc.cx(0, 1)     # CNOT uygula
    sv = Statevector.from_instruction(qc)
    sonuc = max(sv.probabilities_dict(), key=sv.probabilities_dict().get)
    print(f'  |{q0}{q1}⟩ → CNOT → |{sonuc}⟩', end='')
    if q0 == 1:
        print('  ← hedef çevrildi!')
    else:
        print('  ← değişmedi')

---
## 10. SWAP Kapısı

**Ne yapar?** İki qubit'in durumlarını takas eder.

$$\text{SWAP}|01\rangle = |10\rangle, \quad \text{SWAP}|10\rangle = |01\rangle$$

**Analoji:** İki bardağın içindekini birbiriyle değiştirmek.

**İlginç bilgi:** SWAP = 3 tane CNOT kapısından yapılabilir!

In [ ]:
# SWAP: |10⟩ → |01⟩
qc_swap = QuantumCircuit(2)
qc_swap.x(0)         # q0 = |1⟩, q1 = |0⟩
qc_swap.swap(0, 1)   # Takas et

sv_swap = Statevector.from_instruction(qc_swap)
print('SWAP Kapısı:')
print(f'Giriş:  |10⟩ (q0=1, q1=0)')
print(f'Çıkış:  {sv_swap}')
print(f'Sonuç:  |01⟩ (q0=0, q1=1) → Yer değiştirdiler!\n')

display(qc_swap.draw('mpl'))

# SWAP = 3 CNOT
print('\n💡 SWAP kapısı 3 CNOT ile yapılabilir:')
qc_swap3 = QuantumCircuit(2)
qc_swap3.cx(0, 1)
qc_swap3.cx(1, 0)
qc_swap3.cx(0, 1)
display(qc_swap3.draw('mpl'))

---
## 11. Toffoli Kapısı (CCX) — 3 Qubitli

**Ne yapar?** İki kontrol qubit'i de 1 ise, hedefi çevirir. Klasik AND kapısının kuantum karşılığı.

| Kontrol 1 | Kontrol 2 | Hedef (giriş) | Hedef (çıkış) |
|:---------:|:---------:|:--------------:|:--------------:|
| 0 | 0 | x | x |
| 0 | 1 | x | x |
| 1 | 0 | x | x |
| 1 | 1 | x | **NOT x** |

**İsim kökeni:** Tommaso Toffoli (1943-), İtalyan-Amerikan bilgisayar bilimci.

In [ ]:
# Toffoli: |110⟩ → |111⟩
qc_tof = QuantumCircuit(3)
qc_tof.x(0)         # Kontrol 1 = |1⟩
qc_tof.x(1)         # Kontrol 2 = |1⟩
qc_tof.ccx(0, 1, 2) # Toffoli: iki kontrol = 1 → hedefi çevir

sv_tof = Statevector.from_instruction(qc_tof)
print('Toffoli (CCX) Kapısı:')
print(f'Giriş:  |110⟩ (iki kontrol = 1, hedef = 0)')
print(f'Çıkış:  {sv_tof}')
print(f'Sonuç:  |111⟩ → hedef çevrildi!\n')

display(qc_tof.draw('mpl'))

# Kontrol = |10⟩ ise ne olur?
qc_tof2 = QuantumCircuit(3)
qc_tof2.x(0)          # Kontrol 1 = |1⟩
                       # Kontrol 2 = |0⟩ (varsayılan)
qc_tof2.ccx(0, 1, 2)
sv_tof2 = Statevector.from_instruction(qc_tof2)
print(f'\nSadece 1 kontrol = 1 ise:')
print(f'Giriş:  |100⟩ → Çıkış: {sv_tof2}')
print(f'Hedef değişmedi! (İkisi de 1 olmalı)')

---
## 12. Bell Durumu & Dolanıklık ⭐

**H + CNOT birleşimi = Dolanıklık!**

Bu, kuantum bilgisayarların en güçlü özelliğini yaratır.

$$|00\rangle \xrightarrow{H \otimes I} \frac{|00\rangle + |10\rangle}{\sqrt{2}} \xrightarrow{\text{CNOT}} \frac{|00\rangle + |11\rangle}{\sqrt{2}} = |\Phi^+\rangle$$

In [ ]:
# Bell Durumu oluşturma — adım adım

print('='*60)
print('⭐ Bell Durumu: Dolanık Qubit Çifti Oluşturma')
print('='*60)

# Adım 0: Başlangıç
qc0 = QuantumCircuit(2)
sv0 = Statevector.from_instruction(qc0)
print(f'\n📍 Adım 0 — Başlangıç: {sv0}')
print(f'   Durum: |00⟩ (iki qubit de 0)')

# Adım 1: H kapısı
qc1 = QuantumCircuit(2)
qc1.h(0)
sv1 = Statevector.from_instruction(qc1)
print(f'\n📍 Adım 1 — H kapısı q0 ya: {sv1}')
print(f'   Durum: (|00⟩ + |10⟩)/√2 → q0 süperpozisyonda, q1 hâlâ |0⟩')

# Adım 2: CNOT
qc2 = QuantumCircuit(2)
qc2.h(0)
qc2.cx(0, 1)
sv2 = Statevector.from_instruction(qc2)
print(f'\n📍 Adım 2 — CNOT (q0→q1): {sv2}')
print(f'   Durum: (|00⟩ + |11⟩)/√2 → BELL DURUMU!')
print(f'   Ya ikisi 0, ya ikisi 1. Asla biri 0 diğeri 1 çıkmaz!')

# Devre şeması
print('\n📐 Devre Şeması:')
display(qc2.draw('mpl'))

In [ ]:
# Bell durumunu ölçelim — 1000 kez simüle et
from qiskit_aer import AerSimulator

qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0, 1], [0, 1])

# Simüle et
simulator = AerSimulator()
result = simulator.run(qc_bell, shots=1000).result()
counts = result.get_counts()

print('🎲 Bell Durumu — 1000 Ölçüm Sonucu:')
print(f'   {counts}')
print(f'\n💡 Sadece "00" ve "11" çıktı!')
print(f'   "01" ve "10" ASLA çıkmaz — bu dolanıklığın kanıtı.')

# Histogram
fig, ax = plt.subplots(figsize=(8, 4))
states = ['00', '01', '10', '11']
values = [counts.get(s, 0) for s in states]
colors = ['#10B981' if v > 0 else '#E2E8F0' for v in values]
ax.bar(states, values, color=colors, edgecolor='#1E293B', linewidth=1.5)
ax.set_title('Bell Durumu — 1000 Ölçüm', fontsize=14, fontweight='bold')
ax.set_ylabel('Sayı')
for i, (s, v) in enumerate(zip(states, values)):
    ax.text(i, v + 15, str(v), ha='center', fontsize=13, fontweight='bold',
           color='#10B981' if v > 0 else '#94A3B8')
plt.tight_layout()
plt.show()

In [ ]:
# 4 Bell durumunun hepsi
print('='*60)
print('4 Bell Durumu')
print('='*60)

bell_devreleri = {
    '|Φ+⟩ = (|00⟩+|11⟩)/√2': [],
    '|Φ−⟩ = (|00⟩−|11⟩)/√2': ['z'],
    '|Ψ+⟩ = (|01⟩+|10⟩)/√2': ['x'],
    '|Ψ−⟩ = (|01⟩−|10⟩)/√2': ['x', 'z'],
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (isim, ekstra_kapilar) in zip(axes, bell_devreleri.items()):
    qc = QuantumCircuit(2)
    for k in ekstra_kapilar:
        getattr(qc, k)(0)  # q0'a X veya Z uygula
    qc.h(0)
    qc.cx(0, 1)
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities_dict()
    states_all = ['00', '01', '10', '11']
    values = [probs.get(s, 0) for s in states_all]
    colors = ['#10B981' if v > 0.01 else '#E2E8F0' for v in values]
    ax.bar(states_all, values, color=colors, edgecolor='#1E293B')
    ax.set_ylim(0, 0.7)
    ax.set_title(isim, fontsize=11, fontweight='bold')

plt.suptitle('4 Bell Durumu — Olasılık Dağılımları', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Her Bell durumunda sadece 2 sonuç mümkün (diğer 2 imkansız).')
print('   Bu 4 durum, 2-qubitli uzayın dolanık bazıdır.')

---
## 13. Kapı Özet Tablosu

In [ ]:
# Büyük özet tablosu
print('='*80)
print('📋 KUANTUM KAPILARI ÖZET TABLOSU')
print('='*80)
print()
print(f'{"Kapı":<12} {"Qubit":<6} {"İşlev":<35} {"Matris":<25}')
print('-'*80)

kapilar = [
    ('I',        '1', 'Hiçbir şey yapmaz',              '[[1,0],[0,1]]'),
    ('X (NOT)',   '1', '|0⟩↔|1⟩ çevirir',               '[[0,1],[1,0]]'),
    ('Y',        '1', 'Bit + faz çevirir',              '[[0,-i],[i,0]]'),
    ('Z',        '1', '|1⟩ fazını çevirir',             '[[1,0],[0,-1]]'),
    ('H',        '1', 'Süperpozisyon yaratır',          '1/√2 [[1,1],[1,-1]]'),
    ('S (√Z)',    '1', '90° faz ekler',                 '[[1,0],[0,i]]'),
    ('T (π/8)',   '1', '45° faz ekler',                 '[[1,0],[0,e^iπ/4]]'),
    ('Rx(θ)',    '1', 'X ekseni döndürme',              'cos-isin matrisi'),
    ('Ry(θ)',    '1', 'Y ekseni döndürme',              'cos-sin matrisi'),
    ('Rz(θ)',    '1', 'Z ekseni döndürme',              'e^-iθ/2 matrisi'),
    ('CNOT (CX)', '2', 'Kontrollü NOT',                 '4×4 matris'),
    ('SWAP',     '2', 'İki qubit takas',                '4×4 matris'),
    ('CCX',      '3', 'Toffoli (çift kontrollü NOT)',   '8×8 matris'),
]

for kapi, qubit, islev, matris in kapilar:
    print(f'{kapi:<12} {qubit:<6} {islev:<35} {matris:<25}')

print()
print('⭐ Evrensel Kapı Seti: {H, T, CNOT}')
print('   Bu üç kapıyla HERHANGİ bir kuantum işlemi yapılabilir!')

In [ ]:
# Tüm tek-qubit kapılarını Bloch küresinde göster
print('🔵 Tüm Kapıların |0⟩ Üzerindeki Etkisi — Bloch Küresi\n')

tek_kapilar = [
    ('Başlangıç |0⟩', lambda qc: None),
    ('X (NOT)',       lambda qc: qc.x(0)),
    ('Y',             lambda qc: qc.y(0)),
    ('Z',             lambda qc: qc.z(0)),
    ('H',             lambda qc: qc.h(0)),
    ('S',             lambda qc: [qc.h(0), qc.s(0)]),
]

for isim, kapi_fn in tek_kapilar:
    qc = QuantumCircuit(1)
    kapi_fn(qc)
    sv = Statevector.from_instruction(qc)
    print(f'  {isim}: {sv}')
    display(plot_bloch_multivector(sv))

---
## 🎓 Sonuç

Bu not defterinde öğrendiklerimiz:

1. **Pauli kapıları** (X, Y, Z): Qubit'i çevirme ve faz ekleme
2. **Hadamard** (H): Süperpozisyon yaratma — kuantumun anahtarı
3. **Faz kapıları** (S, T): İnce faz kontrolü
4. **Döndürme kapıları** (Rx, Ry, Rz): QML'de veri kodlama
5. **CNOT**: 2-qubit kontrol — dolanıklığın temeli
6. **Bell durumu**: H + CNOT = dolanıklık!
7. **Toffoli**: 3-qubitli klasik AND karşılığı

**Sıradaki adım:** Bu kapıları kullanarak kuantum algoritmaları (Grover, VQE) ve kuantum makine öğrenmesi devreleri oluşturacağız.

---
*Hazırlayan: Dr. Buket Toptaş — Kuantum Makine Öğrenmesi Dersi*